# job2：在 AIDP Notebook 中训练 GBDT

In [347]:
import sys
import pprint
from pyspark.sql import functions as F

DEFAULT_STEP2_OUTPUT_BASE = "/Volumes/data_science_poc/default/data_science_poc/output/step2"
DEFAULT_STANDALONE_STEP2_OUTPUT = f"{DEFAULT_STEP2_OUTPUT_BASE}/20260920060457"

workflow_utils = globals().get("oidlUtils")

step2_output_base = DEFAULT_STEP2_OUTPUT_BASE
step2_output = DEFAULT_STANDALONE_STEP2_OUTPUT

if workflow_utils is not None:
    try:
        step2_output_base = workflow_utils.parameters.getParameter(
            "step2_output_base",
            DEFAULT_STEP2_OUTPUT_BASE,
        )
        step2_output = workflow_utils.parameters.getTaskValue(
            "data_science_job1",
            "step2_output_path",
            DEFAULT_STANDALONE_STEP2_OUTPUT,
        )
    except Exception as exc:
        print(f"workflow parameters unavailable; using standalone path: {type(exc).__name__}")

step2_output = step2_output.rstrip("/") + "/"
print(f"reading step2 data from: {step2_output}")

df = (spark.read.option("header", True)
            .option("encoding", "UTF-8")
            .option("multiLine", True)
            .option("escape", '"')
            .csv(step2_output)
    )

df_clean = df.select(
    F.coalesce(F.col("hashid"), F.lit("")).cast("string").alias("hashid"),
    F.coalesce(F.col("user_query"), F.lit("")).cast("string").alias("user_query"),
    F.coalesce(F.col("docids"), F.lit("")).cast("string").alias("docids"),
    F.col("re_score").cast("double").alias("re_score"),
    F.col("label_score").cast("double").alias("label_score_double")
)

invalid_re_count = df_clean.filter(
    F.col("re_score").isNull() | (F.col("re_score") < 1) | (F.col("re_score") > 100)
).count()
if invalid_re_count > 0:
    raise ValueError("re_score必须是1到100之间的非空数值")

df_clean = df_clean.withColumn(
    "is_int",
    (F.col("label_score_double").isNotNull()) &
    (F.col("label_score_double") == F.floor(F.col("label_score_double")))
)
if df_clean.filter(~F.col("is_int")).count() > 0:
    raise ValueError("label_score必须是整数")

df_clean = df_clean.withColumn("label_score", F.col("label_score_double").cast("int")) \
                   .drop("label_score_double", "is_int")

allowed_labels = [-1, 0, 1, 2, 3]
invalid_labels_df = df_clean.filter(~F.col("label_score").isin(allowed_labels))
if invalid_labels_df.count() > 0:
    invalid_labels = sorted([row["label_score"] for row in invalid_labels_df.select("label_score").distinct().collect()])
    raise ValueError(f"label-score存在超出-1到3范围的值: {invalid_labels}")

label_counts = df_clean.groupBy("label_score").count().collect()
if len(label_counts) < 2:
    raise ValueError("至少需要两个不同的label_score类别")
min_count = min([row["count"] for row in label_counts])
if min_count < 3:
    raise ValueError("每个label_score类别至少需要3条数据")

rows = df_clean.select("hashid", "user_query", "docids", "re_score", "label_score").collect()
records = [row.asDict(recursive=True) for row in rows]

feature_records = [
    {
        "user_query": str(record["user_query"]),
        "docids": str(record["docids"]),
        "re_score": float(record["re_score"]),
    }
    for record in records
]
target_values = [float(record["label_score"]) for record in records]

print(f"数据维度 (行数, 特征数): ({len(feature_records)}, 3)")
print("\n前 5 行数据预览:")
pprint.pprint(records[:5])


reading step2 data from: /Volumes/data_science_poc/default/data_science_poc/output/step2/20260920060457/
Python executable: /opt/dataflow/python/bin/python3


数据维度 (行数, 特征数): (30, 3)

前 5 行数据预览:
[{"docids": "在登录页点击“忘记密码”，输入注册邮箱获取验证码，验证通过后设置新密码即可。",
  "hashid": "a7K2mP9xQ4Ld",
  "label_score": 3,
  "re_score": 52.0,
  "user_query": "如何重置忘记的登录密码？"},
 {"docids": "现货订单通常会在付款成功后的24小时内发货；节假日或预售商品以页面标注时间为准。",
  "hashid": "b3N8vR1cT6Yq",
  "label_score": 3,
  "re_score": 16.0,
  "user_query": "订单付款后多久可以发货？"},
 {"docids": "进入“我的订单”，选择对应订单并点击“申请发票”，填写发票抬头和邮箱后提交，电子发票会发送至该邮箱。",
  "hashid": "c9H4sW7jM2Ua",
  "label_score": 3,
  "re_score": 54.0,
  "user_query": "怎么申请电子发票？"},
 {"docids": "会员积分自获得之日起有效期为12个月，到期未使用的积分会自动失效。",
  "hashid": "d5F1kX8pL3Ze",
  "label_score": 3,
  "re_score": 92.0,
  "user_query": "会员积分的有效期是多久？"},
 {"docids": "请在签收后48小时内拍摄商品、外包装和快递面单照片，并在订单详情中提交售后申请，我们会审核后安排补发或退款。",
  "hashid": "e2Q7nB4tV9Gs",
  "label_score": 3,
  "re_score": 40.0,
  "user_query": "收到商品后发现破损怎么办？"}]


#### 定义模型训练和注册的方法

In [350]:
import math
import random

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.ensemble import GradientBoostingRegressor

# 保留原注册模型名，避免影响已有下游引用；训练器改为集群内置 sklearn GBDT
REGISTERED_MODEL_NAME = "default.default.lgbm_demo1_test"
mlflow.set_experiment("lgbm_demo1_experiment_test")

print("特征记录数量:", len(feature_records))
print("标签数量:", len(target_values))

# 纯 Python 切分
indices = list(range(len(feature_records)))
random.Random(42).shuffle(indices)
test_count = max(1, int(round(len(indices) * 0.2)))
test_indices = set(indices[:test_count])

X_train = [record for index, record in enumerate(feature_records) if index not in test_indices]
X_test = [record for index, record in enumerate(feature_records) if index in test_indices]
y_train = [value for index, value in enumerate(target_values) if index not in test_indices]
y_test = [value for index, value in enumerate(target_values) if index in test_indices]

def select_user_query(records):
    return [record["user_query"] for record in records]

def select_docids(records):
    return [record["docids"] for record in records]

def select_re_score(records):
    return [[record["re_score"]] for record in records]

def build_preprocessor(max_tfidf_features=256):
    return FeatureUnion([
        ("query_tfidf", Pipeline([
            ("select", FunctionTransformer(select_user_query, validate=False)),
            ("tfidf", TfidfVectorizer(max_features=max_tfidf_features)),
        ])),
        ("doc_tfidf", Pipeline([
            ("select", FunctionTransformer(select_docids, validate=False)),
            ("tfidf", TfidfVectorizer(max_features=max_tfidf_features)),
        ])),
        ("re_score", FunctionTransformer(select_re_score, validate=False)),
    ])

def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else matrix

def train_and_register_gbdt(run_name, pipeline, params_to_log):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params_to_log)

        print(f"正在训练 sklearn GBDT 方案 [{run_name}] ...")
        pipeline.fit(X_train, y_train)

        y_pred = [float(value) for value in pipeline.predict(X_test)]
        squared_errors = [(actual - predicted) ** 2 for actual, predicted in zip(y_test, y_pred)]
        rmse = math.sqrt(sum(squared_errors) / len(squared_errors))
        target_mean = sum(y_test) / len(y_test)
        residual_sum = sum(squared_errors)
        total_sum = sum((actual - target_mean) ** 2 for actual in y_test)
        r2 = 0.0 if total_sum == 0 else 1.0 - residual_sum / total_sum

        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            registered_model_name=REGISTERED_MODEL_NAME,
        )

        print(f"[{run_name}] 完成 -> RMSE: {rmse:.4f}, R2: {r2:.4f} | Run ID: {run.info.run_id}")
        return run.info.run_id


特征记录数量: 30
标签数量: 30


训练方案一：sklearn GBDT 基线模型

In [351]:
# ----- 方案 1：sklearn GBDT 基线模型 -----
pipeline_v1 = Pipeline([
    ("preprocessor", build_preprocessor(max_tfidf_features=128)),
    ("to_dense", FunctionTransformer(to_dense, accept_sparse=True)),
    ("model", GradientBoostingRegressor(
        n_estimators=50,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
    )),
])

train_and_register_gbdt(
    run_name="sklearn_gbdt_baseline",
    pipeline=pipeline_v1,
    params_to_log={
        "model_type": "sklearn.GradientBoostingRegressor",
        "n_estimators": 50,
        "learning_rate": 0.1,
        "max_depth": 3,
        "max_tfidf_features": 128,
    },
)


正在训练 sklearn GBDT 方案 [sklearn_gbdt_baseline] ...


[sklearn_gbdt_baseline] 完成 -> RMSE: 0.9304, R2: -0.5582 | Run ID: f460549294cc41659c3a2b4df3ff1961
🏃 View run sklearn_gbdt_baseline at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/61/runs/f460549294cc41659c3a2b4df3ff1961
🧪 View experiment at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/61


"f460549294cc41659c3a2b4df3ff1961"

#### 训练方案二：更多文本特征 + 随机子采样

In [352]:
# ----- 方案 2：更多文本特征 + 随机子采样 -----
pipeline_v2 = Pipeline([
    ("preprocessor", build_preprocessor(max_tfidf_features=512)),
    ("to_dense", FunctionTransformer(to_dense, accept_sparse=True)),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        max_features=0.8,
        random_state=42,
    )),
])

train_and_register_gbdt(
    run_name="sklearn_gbdt_subsampled",
    pipeline=pipeline_v2,
    params_to_log={
        "model_type": "sklearn.GradientBoostingRegressor",
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 3,
        "subsample": 0.8,
        "max_features": 0.8,
        "max_tfidf_features": 512,
    },
)


正在训练 sklearn GBDT 方案 [sklearn_gbdt_subsampled] ...


[sklearn_gbdt_subsampled] 完成 -> RMSE: 0.9809, R2: -0.7320 | Run ID: 52ff50f9c816486dbabf41555a4ec352
🏃 View run sklearn_gbdt_subsampled at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/61/runs/52ff50f9c816486dbabf41555a4ec352
🧪 View experiment at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/61


"52ff50f9c816486dbabf41555a4ec352"

#### 训练方案三：使用浅树+叶节点约束

In [364]:
# ----- 方案 3：浅树 + 叶节点约束，降低小样本过拟合 -----
pipeline_v3 = Pipeline([
    ("preprocessor", build_preprocessor(max_tfidf_features=256)),
    ("to_dense", FunctionTransformer(to_dense, accept_sparse=True)),
    ("model", GradientBoostingRegressor(
        n_estimators=80,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=3,
        subsample=0.8,
        random_state=42,
    )),
])

train_and_register_gbdt(
    run_name="sklearn_gbdt_regularized",
    pipeline=pipeline_v3,
    params_to_log={
        "model_type": "sklearn.GradientBoostingRegressor",
        "n_estimators": 80,
        "learning_rate": 0.03,
        "max_depth": 2,
        "min_samples_leaf": 3,
        "subsample": 0.8,
        "max_tfidf_features": 256,
    },
)


正在训练 sklearn GBDT 方案 [sklearn_gbdt_regularized] ...


[sklearn_gbdt_regularized] 完成 -> RMSE: 0.7818, R2: -0.1003 | Run ID: 42c063c4b309459797916c83e501fb6c
🏃 View run sklearn_gbdt_regularized at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/61/runs/42c063c4b309459797916c83e501fb6c
🧪 View experiment at: https://c1.df.aidp.us-chicago-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/20240831/dataLakes/ocid1.aidataplatform.oc1.us-chicago-1.amaaaaaaak7gbriatbmy3ojjkkpmwigz2zkrn425b6ibxgsggoy5z4jva6sq/workspaces/a00bcc21-422c-452c-88e0-f2d39f0faeba/mlops/#/experiments/61


"42c063c4b309459797916c83e501fb6c"

#### 给模型设置生成别名（@champion）

In [353]:
# =====================================================================
# 5. 设置生产别名 (@champion)
# =====================================================================
update_model_aliases = True
if workflow_utils is not None:
    try:
        update_model_aliases = workflow_utils.parameters.getParameter(
            "update_model_aliases",
            "true",
        ).strip().lower() in {"true", "1", "yes", "y"}
    except Exception:
        update_model_aliases = True

if update_model_aliases:
    client = MlflowClient()
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="champion",
        version="2"
    )
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="challenger",
        version="3"
    )
    print("所有 3 个 sklearn GBDT 版本训练完成并注册！@champion -> Version 2, @challenger -> Version 3")
else:
    print("模型训练与注册完成；本次验证运行跳过 champion/challenger alias 更新")


所有 3 个 sklearn GBDT 版本训练完成并注册！@champion -> Version 2, @challenger -> Version 3


#### 使用MLFlow下载Artifact

In [376]:
from mlflow.tracking import MlflowClient

# 模型版本
MODEL_VERSION = "18"

client = MlflowClient()

model_version = client.get_model_version(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
)

print("Model name:", model_version.name)
print("Model version:", model_version.version)
print("Run ID:", model_version.run_id)
print("Model Source:",model_version.source)

Model name: default.default.lgbm_demo1_test
Model version: 18
Run ID: 42c063c4b309459797916c83e501fb6c
Artifact source: mlops-artifact-store:/Volumes/default/default/AIDPModels/5962058d83dd4df19dc24bf5a52c36da/artifacts
volume_path: /Volumes/default/default/AIDPModels/5962058d83dd4df19dc24bf5a52c36da/artifacts


In [403]:
import os
import mlflow

# 下载到github上
download_root = f"/Workspace/git-poc/ml-poc/download/{REGISTERED_MODEL_NAME}/{MODEL_VERSION}"
# 创建路径
os.makedirs(download_root, exist_ok=True)
# 获取source
artifact_source = model_version.source

# 使用下载SDK
downloaded_path = mlflow.artifacts.download_artifacts(
    artifact_uri=artifact_source,
    dst_path=download_root,
)

print("下载后的本地路径:")
print(downloaded_path)

下载后的本地路径:
/Workspace/git-poc/ml-poc/download/default.default.lgbm_demo1_test/18/artifacts
